## Developer environment for silver layer

In [50]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Átállítjuk az Ivy cache-t a saját home könyvtáradba, ahol van írási jogod
spark = (
    SparkSession.builder.appName("DataLakeExperiment")
    .master("local[*]")
    .config(
        "spark.jars.ivy", "/home/azureuser/.ivy2"
    )  # <--- Itt a változtatás
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")



## Reading

In [55]:


stop_df = spark.read.format("parquet").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/dim_stop.parquet"
)

route_df = spark.read.format("parquet").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/dim_route.parquet"
)

trip_df = spark.read.format("parquet").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/dim_trip.parquet"
)

# Must be a streaming reading in the production environment 

vhc_positions_df = spark.read.format("delta").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/vehicle_positions"
)

trip_updates_df = spark.read.format("delta").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/trip_updates"
)



## Cleaning and transform Vehicle positions

In [56]:
## Imports
import pyspark.sql.functions as F


## Vehicle cleaning from Null values
clean_vhc_positions_df = vhc_positions_df.filter(
    (F.col("route_id").isNotNull()) & (F.trim(F.col("route_id")) != "") &
    (F.col("trip_id").isNotNull()) & (F.trim(F.col("trip_id")) != "") &
    (F.col("stop_id").isNotNull()) & (F.trim(F.col("stop_id")) != "")
)



## Create the vehicle dimension from the vehicle_positions to build the star schema 

In [57]:
# Create the new dim and drop duplicates
vehicle_df = clean_vhc_positions_df.select("vehicle_id", "vehicle_label","license_plate" ).dropDuplicates()

# Remove the colums that we already loaded to another dim
clean_vhc_positions_df = clean_vhc_positions_df.drop(
     "vehicle_label", "license_plate"
)



## Cleaning and transform trip updates

In [58]:
## Imports
import pyspark.sql.functions as F


## Trip updates cleaning from Null values
clean_trip_updates_df = trip_updates_df.filter(
    (F.col("route_id").isNotNull()) & (F.trim(F.col("route_id")) != "") &
    (F.col("trip_id").isNotNull()) & (F.trim(F.col("trip_id")) != "") &
    (F.col("stop_id").isNotNull()) & (F.trim(F.col("stop_id")) != "")
)


## Join

In [59]:
import pyspark.sql.functions as F

# Create the fact table from vehicle positions and trip updates
fact_df = clean_vhc_positions_df.join(
    clean_trip_updates_df,
    (clean_vhc_positions_df["trip_id"] == clean_trip_updates_df["trip_id"])
    & (clean_vhc_positions_df["route_id"] == clean_trip_updates_df["route_id"])  
    & (clean_vhc_positions_df["stop_id"] == clean_trip_updates_df["stop_id"]),
    "left_outer",
).select(
    clean_vhc_positions_df["trip_id"],
    clean_vhc_positions_df["route_id"],
    clean_vhc_positions_df["stop_id"],
    clean_vhc_positions_df["vehicle_id"],
    clean_vhc_positions_df["latitude"],
    clean_vhc_positions_df["longitude"],
    clean_vhc_positions_df["speed"],
    clean_vhc_positions_df["bearing"],
    clean_vhc_positions_df["current_status"],
    clean_vhc_positions_df["vehicle_timestamp"],
    clean_trip_updates_df["arrival_time"],
    clean_trip_updates_df["departure_time"],
    clean_trip_updates_df["arrival_uncertainty"],
    clean_trip_updates_df["ingested_at"],
)



## Write the datas into the silver layer of the data lake

In [60]:
route_df.write.mode("overwrite").parquet("/home/azureuser/BKK-Streaming_pipeline/data-lake/silver/route")
stop_df.write.mode("overwrite").parquet("/home/azureuser/BKK-Streaming_pipeline/data-lake/silver/stop")
trip_df.write.mode("overwrite").parquet("/home/azureuser/BKK-Streaming_pipeline/data-lake/silver/trip")
vehicle_df.write.mode("overwrite").parquet("/home/azureuser/BKK-Streaming_pipeline/data-lake/silver/vehicle")
# Must be streaming writing

fact_df.write.format("delta").mode("append").save("/home/azureuser/BKK-Streaming_pipeline/data-lake/silver/fact")